# 01 · Recolección de titulares: La República y El Comercio

**Pregunta:** ¿Podemos juntar 200 titulares de cada diario, en cuatro categorías, para comparar y entrenar un clasificador?

Opción **C (comparada)** de la tarea: 200 de La República + 200 de El Comercio.
Este notebook se corre **una vez al día**: cada corrida agrega las notas nuevas a `datos/noticias.csv` sin repetir ninguna.

In [1]:
import time
from pathlib import Path
from urllib.robotparser import RobotFileParser
import xml.etree.ElementTree as ET

import pandas as pd
import requests

## 1. Reglas del juego: robots.txt

Antes de pedir nada, revisamos qué permite cada diario. Ambos publican feeds RSS y los listan como canales oficiales.

In [2]:
feeds_larepublica = {
    "home": "https://larepublica.pe/rss/home.xml",
    "politica": "https://larepublica.pe/rss/politica.xml",
    "economia": "https://larepublica.pe/rss/economia.xml",
    "deportes": "https://larepublica.pe/rss/deportes.xml",
    "ciencia": "https://larepublica.pe/rss/ciencia.xml",
}

plantilla_elcomercio = "https://elcomercio.pe/arc/outboundfeeds/google-news-feed/category/{}/?outputType=xml"
feeds_elcomercio = {
    "politica": plantilla_elcomercio.format("politica"),
    "economia": plantilla_elcomercio.format("economia"),
    "deporte-total": plantilla_elcomercio.format("deporte-total"),
    "tecnologia": plantilla_elcomercio.format("tecnologia"),
}

In [3]:
cabeceras = {
    "User-Agent": "TareaDiplomadoPUCP/1.0 (uso academico; contacto: huarhuarojas@gmail.com)"
}
segundos_de_pausa = 1.5

In [4]:
# Pedimos permiso a robots.txt para cada feed que vamos a usar.
# Si algún feed saliera como False, no se descarga.
permisos = []

for medio, sitio, feeds in [
    ("La Republica", "https://larepublica.pe", feeds_larepublica),
    ("El Comercio", "https://elcomercio.pe", feeds_elcomercio),
]:
    lector_robots = RobotFileParser()
    lector_robots.set_url(sitio + "/robots.txt")
    lector_robots.read()
    for nombre, url in feeds.items():
        permitido = lector_robots.can_fetch(cabeceras["User-Agent"], url)
        permisos.append({"medio": medio, "feed": nombre, "permitido": permitido})

pd.DataFrame(permisos)

,medio,feed,permitido
0,La Republica,home,True
1,La Republica,politica,True
2,La Republica,economia,True
3,La Republica,deportes,True
4,La Republica,ciencia,True
5,El Comercio,politica,True
6,El Comercio,economia,True
7,El Comercio,deporte-total,True
8,El Comercio,tecnologia,True


## 2. Leer un feed

Un feed RSS es un XML con una lista de `<item>`. Primero lo vemos con **un** feed.

In [5]:
respuesta = requests.get(feeds_larepublica["politica"], headers=cabeceras, timeout=30)
respuesta.status_code

200

In [6]:
raiz = ET.fromstring(respuesta.content.strip())
notas = raiz.findall("./channel/item")
len(notas)

29

In [7]:
primera_nota = notas[0]

# findtext devuelve el texto de una etiqueta; strip() quita espacios y saltos de línea sobrantes
print("titular:", primera_nota.findtext("title").strip())
print("bajada: ", primera_nota.findtext("description").strip())
print("url:    ", primera_nota.findtext("link").strip())
print("fecha:  ", primera_nota.findtext("pubDate").strip())

titular: Rafael Rey confiesa que le ofrecieron la embajada del Perú en España: "No hay decisión tomada”
bajada:  El canciller Carlos Espá ya había deslizado la posibilidad de que Rey sea nombrado como representante diplomático en el Perú.
url:     https://larepublica.pe/politica/2026/10/14/rafael-rey-confiesa-que-le-ofrecieron-la-embajada-del-peru-en-espana-no-hay-decision-tomada-hnews-465528
fecha:   Thu, 15 Oct 2026 03:18:14 GMT


## 3. Repetir para los nueve feeds

Esto se repite nueve veces, así que ahora sí conviene una función.
Regla del enunciado: pausa de 1 a 2 segundos y **máximo un reintento**.

In [8]:
def descargar_feed(medio, url):
    respuesta = None
    for intento in range(2):
        try:
            respuesta = requests.get(url, headers=cabeceras, timeout=30)
            if respuesta.status_code == 200:
                break
        except requests.RequestException:
            respuesta = None
        time.sleep(segundos_de_pausa)

    if respuesta is None or respuesta.status_code != 200:
        print("No se pudo descargar:", url)
        return []

    raiz = ET.fromstring(respuesta.content.strip())
    filas = []
    for nota in raiz.findall("./channel/item"):
        filas.append({
            "titular": (nota.findtext("title") or "").strip(),
            "bajada": (nota.findtext("description") or "").strip(),
            "medio": medio,
            "url": (nota.findtext("link") or "").strip(),
            "fecha": (nota.findtext("pubDate") or "").strip(),
        })
    return filas

In [9]:
todas_las_notas = []

for medio, feeds in [("La Republica", feeds_larepublica), ("El Comercio", feeds_elcomercio)]:
    for nombre, url in feeds.items():
        notas_del_feed = descargar_feed(medio, url)
        todas_las_notas = todas_las_notas + notas_del_feed
        print(medio, nombre, len(notas_del_feed))
        time.sleep(segundos_de_pausa)

crudo = pd.DataFrame(todas_las_notas)
crudo.shape

La Republica home 200


La Republica politica 29


La Republica economia 12


La Republica deportes 15


La Republica ciencia 14


El Comercio politica 43


El Comercio economia 83


El Comercio deporte-total 71


El Comercio tecnologia 18


(485, 5)

## 4. Sección original y categoría estandarizada

La sección es el primer tramo de la URL (`/politica/2026/...`). El mapeo a las cuatro categorías oficiales es explícito.

In [10]:
# split("/")[3] toma el primer tramo después del dominio: https://dominio/SECCION/...
crudo["seccion_original"] = crudo["url"].str.split("/").str[3]
crudo["seccion_original"].value_counts()

seccion_original
economia            102
politica             92
deporte-total        70
sociedad             37
espectaculos         32
ciencia              28
deportes             27
mundo                21
tecnologia           18
opinion              16
latinos-en-usa       13
entretenimiento       8
cultural              4
cine-series           4
ecdata                4
cultura-asiatica      2
verificador           2
servicios             1
publirreportajes      1
mastergrama           1
carlincatura          1
horoscopo             1
Name: count, dtype: int64

In [11]:
mapa_de_categorias = {
    "politica": "politica",
    "economia": "economia",
    "deportes": "deportes",
    "deporte-total": "deportes",
    "ciencia": "ciencia_tecnologia",
    "tecnologia": "ciencia_tecnologia",
}

crudo["etiqueta"] = crudo["seccion_original"].map(mapa_de_categorias)

## 5. Lo que no encaja se descarta y se reporta

Sociedad, espectáculos, mundo, etc. no son una de las cuatro categorías. No inventamos una quinta.

In [12]:
descartadas = crudo[crudo["etiqueta"].isna()]
print("Descartadas:", len(descartadas), "de", len(crudo))
descartadas.groupby(["medio", "seccion_original"]).size().sort_values(ascending=False)

Descartadas: 148 de 485


medio         seccion_original
La Republica  sociedad            37
              espectaculos        32
              mundo               21
              opinion             14
              latinos-en-usa      13
              entretenimiento      8
El Comercio   ecdata               4
La Republica  cine-series          4
              cultural             4
              verificador          2
El Comercio   opinion              2
La Republica  cultura-asiatica     2
              carlincatura         1
              horoscopo            1
              mastergrama          1
              servicios            1
              publirreportajes     1
dtype: int64

In [13]:
notas_utiles = crudo[crudo["etiqueta"].notna()].copy()
notas_utiles.shape

(337, 7)

## 6. Limpieza

El mismo artículo puede salir en dos feeds (por ejemplo, `home` y `politica`). La URL identifica la nota.

In [14]:
# Un diario escribe la zona como "GMT" y el otro como "+0000"; format="mixed" acepta ambos.
# Pasamos a hora de Lima y a AAAA-MM-DD, para que un día sea un día.
fecha_utc = pd.to_datetime(notas_utiles["fecha"], utc=True, format="mixed")
notas_utiles["fecha"] = fecha_utc.dt.tz_convert("America/Lima").dt.strftime("%Y-%m-%d")

In [15]:
notas_utiles = notas_utiles[notas_utiles["titular"] != ""]
notas_utiles = notas_utiles.drop_duplicates(subset="url")
notas_utiles.shape

(278, 7)

In [16]:
columnas = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
notas_de_hoy = notas_utiles[columnas]

## 7. Acumular con lo ya guardado

Cada día entran notas nuevas. Juntamos lo de hoy con el CSV anterior y quitamos repetidas por URL.

In [17]:
ruta_csv = Path("datos/noticias.csv")

if ruta_csv.exists():
    acumulado = pd.read_csv(ruta_csv)
else:
    acumulado = pd.DataFrame(columns=columnas)

antes = len(acumulado)
acumulado = pd.concat([acumulado, notas_de_hoy], ignore_index=True)
acumulado = acumulado.drop_duplicates(subset="url", keep="first")

print("Antes:", antes, "| Después:", len(acumulado), "| Nuevas hoy:", len(acumulado) - antes)

Antes: 399 | Después: 562 | Nuevas hoy: 163


In [18]:
# Sin índice y en UTF-8, para que las tildes sobrevivan
ruta_csv.parent.mkdir(exist_ok=True)
acumulado.to_csv(ruta_csv, index=False, encoding="utf-8")

## 8. La República se queda corta: seguir enlaces entre notas

Los feeds RSS de La República cubren solo unos 2 días y rotan despacio: tras varias corridas seguía lejos de 200.
La decisión, con sus alternativas, está en `BITACORA.md`.

Reglas de esta sección:

- solo notas de las 4 secciones oficiales,
- las mismas reglas de siempre: robots.txt, User-Agent, pausa de 1.5 s y un solo reintento,
- un **tope por sección**, para que deportes no ahogue a economía,
- un **máximo de pedidos**, para no cargar al sitio.

In [19]:
import re
from bs4 import BeautifulSoup

robots_larepublica = RobotFileParser()
robots_larepublica.set_url("https://larepublica.pe/robots.txt")
robots_larepublica.read()

# Una nota de las 4 secciones: /economia/2026/10/13/... o /economia/subseccion/2026/10/13/...
# El enlace puede venir con o sin dominio, entre comillas, dentro del HTML o de un JSON
patron_de_nota = re.compile(
    r'["\'](?:https://larepublica\.pe)?(/(?:politica|economia|deportes|ciencia)/(?:[a-z0-9-]+/)?20\d\d/\d\d/\d\d/[^"\'#?\s\\<]+)'
)

# robots.txt prohíbe paginación, tags y archivo. RobotFileParser no entiende los comodines (*),
# así que además excluimos esas rutas a mano.
rutas_prohibidas = ["/pag/", "/page/", "/tag/", "/archivo/", "/archive/", "/buscador"]

In [20]:
def enlaces_de_notas(html):
    encontrados = set()
    for ruta in patron_de_nota.findall(html):
        prohibida = False
        for fragmento in rutas_prohibidas:
            if fragmento in ruta:
                prohibida = True
        if not prohibida:
            encontrados.add("https://larepublica.pe" + ruta)
    return encontrados

In [21]:
meta_larepublica = 200
tope_por_seccion = 60
maximo_de_pedidos = 250

ya_guardadas = set(acumulado["url"])
cola = []
pedidos = 0

# Semillas 1: las cuatro portadas de sección
for seccion in ["politica", "economia", "deportes", "ciencia"]:
    url_portada = "https://larepublica.pe/" + seccion + "/"
    if robots_larepublica.can_fetch(cabeceras["User-Agent"], url_portada):
        respuesta = requests.get(url_portada, headers=cabeceras, timeout=30)
        pedidos = pedidos + 1
        for url in enlaces_de_notas(respuesta.text):
            if url not in ya_guardadas and url not in cola:
                cola.append(url)
    time.sleep(segundos_de_pausa)

# Semillas 2: las notas que ya tenemos. Las portadas muestran casi lo mismo que el RSS,
# pero cada nota enlaza a otras relacionadas, y ahí aparecen notas que el RSS ya no trae.
semillas = list(acumulado.loc[acumulado["medio"] == "La Republica", "url"])

print("Candidatas en la cola:", len(cola), "| Notas semilla:", len(semillas))

Candidatas en la cola: 0 | Notas semilla: 249


### Recorrer la cola

Cada nota que visitamos aporta su titular, bajada y fecha, y además nos enlaza a otras notas que sumamos a la cola.

In [22]:
notas_nuevas_larepublica = []
visitadas = set()
errores = 0

filas_larepublica = acumulado[acumulado["medio"] == "La Republica"]
conteo_por_seccion = filas_larepublica["seccion_original"].value_counts().to_dict()
total_larepublica = len(filas_larepublica)

while (len(cola) > 0 or len(semillas) > 0) and total_larepublica < meta_larepublica and pedidos < maximo_de_pedidos:
    # Primero las notas nuevas; solo si no quedan, leemos una semilla para sacarle enlaces
    if len(cola) == 0:
        cola.append(semillas.pop(0))

    url = cola.pop(0)
    seccion = url.split("/")[3]
    es_semilla = url in ya_guardadas

    if url in visitadas:
        continue
    visitadas.add(url)

    if not es_semilla and conteo_por_seccion.get(seccion, 0) >= tope_por_seccion:
        continue
    if not robots_larepublica.can_fetch(cabeceras["User-Agent"], url):
        continue

    html = None
    for intento in range(2):
        try:
            respuesta = requests.get(url, headers=cabeceras, timeout=30)
            pedidos = pedidos + 1
            if respuesta.status_code == 200:
                html = respuesta.text
                break
        except requests.RequestException:
            pass
        time.sleep(segundos_de_pausa)
    time.sleep(segundos_de_pausa)

    if html is None:
        errores = errores + 1
        continue

    # Una nota nueva se guarda; una semilla solo se lee para sacarle enlaces
    if not es_semilla:
        # El titular y la bajada vienen en las etiquetas meta; la fecha, en los datos estructurados
        pagina = BeautifulSoup(html, "html.parser")
        meta_titulo = pagina.find("meta", property="og:title")
        meta_bajada = pagina.find("meta", property="og:description")
        busqueda_fecha = re.search(r'"datePublished":"(\d{4}-\d{2}-\d{2})', html)

        if meta_titulo is None or busqueda_fecha is None:
            errores = errores + 1
            continue

        bajada = ""
        if meta_bajada is not None:
            bajada = meta_bajada.get("content", "").strip()

        notas_nuevas_larepublica.append({
            "titular": meta_titulo.get("content", "").strip(),
            "bajada": bajada,
            "medio": "La Republica",
            "seccion_original": seccion,
            "url": url,
            "fecha": busqueda_fecha.group(1),
            "etiqueta": mapa_de_categorias.get(seccion),
        })
        conteo_por_seccion[seccion] = conteo_por_seccion.get(seccion, 0) + 1
        total_larepublica = total_larepublica + 1

    for enlace in enlaces_de_notas(html):
        if enlace not in ya_guardadas and enlace not in visitadas and enlace not in cola:
            cola.append(enlace)

print("Pedidos:", pedidos, "| Notas nuevas:", len(notas_nuevas_larepublica), "| Errores:", errores)
print("La República ahora:", total_larepublica, "de", meta_larepublica)

Pedidos: 4 | Notas nuevas: 0 | Errores: 0
La República ahora: 249 de 200


In [23]:
# Mismo criterio que antes: sin repetidas por URL y sin etiquetas vacías
nuevas_de_larepublica = pd.DataFrame(notas_nuevas_larepublica, columns=columnas)
nuevas_de_larepublica = nuevas_de_larepublica[nuevas_de_larepublica["etiqueta"].notna()]

acumulado = pd.concat([acumulado, nuevas_de_larepublica], ignore_index=True)
acumulado = acumulado.drop_duplicates(subset="url", keep="first")
acumulado.to_csv(ruta_csv, index=False, encoding="utf-8")
len(acumulado)

562

## 9. ¿Cuánto falta? Revisión del dataset

In [24]:
pd.crosstab(acumulado["medio"], acumulado["etiqueta"], margins=True)

etiqueta,ciencia_tecnologia,deportes,economia,politica,All
medio,,,,,
El Comercio,32,132,88,61,313
La Republica,51,50,67,81,249
All,83,182,155,142,562


In [25]:
meta_por_medio = 200
total_por_medio = acumulado["medio"].value_counts()
faltan = meta_por_medio - total_por_medio
faltan.clip(lower=0)

medio
El Comercio     0
La Republica    0
Name: count, dtype: int64

In [26]:
# ¿Hay valores vacíos? La bajada puede faltar en algunas notas.
acumulado.isna().sum()

titular              0
bajada              14
medio                0
seccion_original     0
url                  0
fecha                0
etiqueta             0
dtype: int64

In [27]:
# Rango de fechas cubierto
acumulado.groupby("medio")["fecha"].agg(["min", "max"])

,min,max
medio,,
El Comercio,2026-10-08,2026-10-14
La Republica,2021-03-09,2026-10-14


## Qué hicimos

| Paso | Código |
|---|---|
| Revisar permisos | `RobotFileParser`, `can_fetch` |
| Identificarnos | `headers={"User-Agent": ...}` |
| Leer un feed RSS | `requests.get` + `ET.fromstring` + `findall("./channel/item")` |
| Repetir con pausa y un reintento | `descargar_feed`, `time.sleep` |
| Sección desde la URL | `str.split("/").str[3]` |
| Mapear a 4 categorías | `.map(mapa_de_categorias)` |
| Descartar lo que no encaja | `etiqueta.isna()` |
| Acumular sin repetir | `pd.concat` + `drop_duplicates(subset="url")` |
| Completar La República | `enlaces_de_notas`, `BeautifulSoup`, cola con tope por sección |
| Guardar | `to_csv(index=False)` |